# De tu audio a un video con letras
Ejecuta los pasos en orden con el botón ▶ de cada sección. No necesitas instalar programas en tu equipo ni escribir código.

**Necesitas:** una cuenta de Google, audio propio o autorizado y el archivo `.whl` que te entrega el responsable del proyecto junto con este cuaderno. No necesitas acceso al repositorio privado ni tokens.

En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, selecciona **T4 GPU**. Su disponibilidad depende de Colab. La instalación y la primera descarga de modelos pueden tardar varios minutos.

**Recorrido:** instalar → subir audio → preparar → escuchar y revisar → generar → descargar. Usa primero un fragmento corto. No ejecutes todo sin revisar las letras.

El audio y los resultados permanecen en esta sesión temporal de Colab: descárgalos antes de cerrarla. No se monta Google Drive. El modelo de alineación tiene licencia CC-BY-NC-4.0; consulta los avisos de terceros del proyecto.

In [ ]:
# @title 1. Instalar el motor (sube el archivo .whl recibido)
import json
import os
import shutil
import signal
import subprocess
import sys
import uuid
import zipfile
from pathlib import Path

from google.colab import files
from IPython.display import Audio, Video, display

if sys.version_info < (3, 12):  # noqa: UP036 - validate the host before installing the package
    raise RuntimeError('Selecciona un entorno de Colab con Python 3.12 o posterior.')
ROOT = Path('/content/karaoke-notebook')
ROOT.mkdir(exist_ok=True)
OUTPUT_DIR = ROOT / 'projects'
ENGINE = Path('/content/karaoke-env/bin/python')

def select_file(folder, extensions, label):
    folder.mkdir(parents=True, exist_ok=True)
    candidates = [p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions]
    if len(candidates) != 1:
        print(label)
        uploaded = files.upload(target_dir=str(folder))
        candidates = [folder / Path(name).name for name in uploaded if Path(name).suffix.lower() in extensions]
    if len(candidates) != 1:
        raise ValueError('Selecciona exactamente un archivo del tipo indicado y repite este paso.')
    print('Archivo seleccionado:', candidates[0].name)
    return candidates[0]

def run_command(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            if process.wait():
                raise RuntimeError('El proceso no terminó correctamente. Revisa el mensaje anterior; no continúes.')
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGINT)
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
            raise

def run_engine(operation, request):
    request_path = ROOT / f'{uuid.uuid4().hex}.request.json'
    result_path = request_path.with_suffix('.result.json')
    request_path.write_text(json.dumps(request), encoding='utf-8')
    run_command([str(ENGINE), '-m', 'karaoke_app.notebook_runner', operation,
                 '--request', str(request_path), '--result', str(result_path)])
    if not result_path.is_file():
        raise RuntimeError('No se recibió un resultado completo. Revisa los mensajes del proceso.')
    return json.loads(result_path.read_text(encoding='utf-8'))

wheel = select_file(ROOT / 'package', {'.whl'}, 'Sube el paquete .whl recibido.')
with zipfile.ZipFile(wheel) as package:
    requirements = package.read('karaoke_app/runtime/requirements.txt')
requirements_path = ROOT / 'requirements.txt'
requirements_path.write_bytes(requirements)
uv = shutil.which('uv')
if uv is None:
    bootstrap = ROOT / 'bootstrap'
    run_command([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap), 'uv==0.13.4'])
    uv = str(bootstrap / 'bin' / 'uv')
run_command([uv, 'venv', '--allow-existing', '--python', sys.executable, str(ENGINE.parent.parent)])
run_command([uv, 'pip', 'install', '--python', str(ENGINE), '--reinstall-package', 'lyrics-generator',
             str(wheel), '-r', str(requirements_path)])
run_command([uv, 'pip', 'check', '--python', str(ENGINE)])
print('Instalación completa. El entorno de Colab no se ha reemplazado.')


In [ ]:
# @title 2. Comprobar que la GPU está lista
if 'run_engine' not in globals():
    raise RuntimeError('Primero completa la instalación del paso 1.')
report = run_engine('diagnose', {})
if not report['cuda_available']:
    raise RuntimeError('No se detectó GPU. Selecciona T4 en el tipo de entorno y vuelve al paso 1.')
if not all(report['modules'].values()) or not all(report['commands'].values()):
    raise RuntimeError('Faltan componentes. Revisa la instalación del paso 1 antes de continuar.')
print('Motor listo. GPU:', report['gpu'])


## Tu canción
Sube **un archivo local** de audio o video y completa el título real, artista e idioma; el álbum es opcional. La búsqueda automática está activada: el motor consulta LRCLIB y después Genius; si no encuentra una referencia válida, recurre a transcripción. Comprueba el origen de la letra en el paso 5.

**No necesitas un TXT.** Si quieres aportar una letra, selecciónala junto al audio en el mismo diálogo; tendrá prioridad sobre la búsqueda. Esta versión no admite URL de YouTube ni cookies. Los archivos son temporales y no se guardan en Drive.

**Ejecuta el paso 3 para mostrar el selector de archivos debajo de la celda.** Acepta un WAV, FLAC, MP3, M4A, OGG, MP4 o WEBM y, opcionalmente, un TXT. Se comprueba el contenido y la presencia de audio, no solo la extensión. Para cambiar de canción, repite el paso 3 y selecciona el nuevo archivo; los proyectos anteriores no se sobrescriben.

In [ ]:
# @title 3. Elegir audio y completar los datos
# @markdown Al ejecutar esta celda aparece el selector. Elige un audio/video y, si quieres, un TXT juntos.
TITULO = '' # @param {type:"string"}
ARTISTA = '' # @param {type:"string"}
ALBUM = '' # @param {type:"string"}
IDIOMA = 'es' # @param ['es', 'en', 'pt', 'fr', 'it', 'de']
BUSCAR_LETRA_EN_LINEA = True # @param {type:"boolean"}
audio = None
reference = None
media_info = None
prepared = None
rendered = None
reviewed_project = None
if 'run_engine' not in globals():
    raise RuntimeError('Primero completa la instalación del paso 1.')
upload_dir = ROOT / 'uploads' / uuid.uuid4().hex
upload_dir.mkdir(parents=True)
print('Selecciona un audio/video. Puedes incluir un TXT de letras, pero no es necesario.')
uploaded = files.upload(target_dir=str(upload_dir))
upload_paths = [str(upload_dir / Path(name).name) for name in uploaded]
del uploaded
selection = run_engine('validate-input', {'paths': upload_paths})
audio = Path(selection['source'])
reference = Path(selection['lyrics_text']) if selection['lyrics_text'] else None
media_info = selection['media']
print('Archivo validado:', audio.name)
print('Formato:', media_info['format'], '| Audio:', media_info['audio_codec'])
print(f"Duración: {media_info['duration_seconds']:.1f} segundos")
print('Letra:', reference.name if reference else 'Búsqueda automática o transcripción, según tu selección')
print('Canción:', TITULO.strip() or audio.stem, '| Artista:', ARTISTA.strip() or 'Sin indicar')
print('Álbum:', ALBUM.strip() or 'Sin indicar', '| Idioma:', IDIOMA)
print('Siguiente paso: separar la voz, obtener la letra y sincronizarla.')


In [ ]:
# @title 4. Preparar el audio y las letras
# @markdown La primera ejecución descarga modelos. Espera al mensaje «Proyecto preparado».
MODELO_TRANSCRIPCION = 'large-v3' # @param ['small', 'medium', 'large-v3']
prepared = None
rendered = None
reviewed_project = None
if not globals().get('audio') or not globals().get('media_info'):
    raise RuntimeError('Primero completa la selección de archivos del paso 3.')
if BUSCAR_LETRA_EN_LINEA and reference is None and (not TITULO.strip() or not ARTISTA.strip()):
    raise ValueError('Completa título y artista en el paso 3 para buscar la letra automáticamente.')
prepared = run_engine('prepare', {
    'source': str(audio), 'output_dir': str(OUTPUT_DIR), 'title': TITULO.strip() or None,
    'artist': ARTISTA.strip() or None, 'album': ALBUM.strip() or None, 'language': IDIOMA,
    'lyrics_text': str(reference) if reference else None, 'use_lrclib': BUSCAR_LETRA_EN_LINEA,
    'whisper_model': MODELO_TRANSCRIPCION,
})
print('Proyecto preparado. Escucha y revisa las letras en el siguiente paso.')


In [ ]:
# @title 5. Escuchar la voz y revisar las letras
MOSTRAR_TIEMPOS_POR_PALABRA = False # @param {type:"boolean"}
if not globals().get('prepared'):
    raise RuntimeError('Primero completa la preparación del paso 4.')
info = run_engine('inspect', {'directory': prepared['directory']})
song = info['song']
print('Canción:', song['title'], '| Artista:', song['artist'] or 'Sin indicar')
print('Álbum:', song['album'] or 'Sin indicar', '| Idioma:', song['language'] or 'Sin indicar')
evidence = info['review']
sources = {'lrclib': 'LRCLIB', 'genius': 'Genius', 'txt': 'TXT aportado',
           'whisper': 'Transcripción automática (Whisper)'}
print('Origen de la letra:', sources.get(evidence['reference_source'], 'Sin información'))
print('Motor de sincronización:', evidence['alignment_engine'] or 'Sin información')
print('Comprobación de anclas:', evidence['anchor_status'] or 'Sin información')
if evidence['reference_source'] == 'whisper':
    print('Se utilizó transcripción, no una letra recuperada de un proveedor. Revisa posibles errores.')
if evidence['needs_review'] or evidence['anchor_status'] == 'insufficient_anchors':
    print('Atención: la evidencia automática no basta para confirmar los tiempos. Revísalos escuchando la voz.')
display(Audio(filename=info['vocals']))
print('Inicio → Fin (segundos) | Letra')
for line in info['timed_lines']:
    print(f"{line['start_ms'] / 1000:8.2f} → {line['end_ms'] / 1000:8.2f} | {line['text']}")
    if MOSTRAR_TIEMPOS_POR_PALABRA:
        for word in line.get('words', []):
            print(f"    {word['start_ms'] / 1000:.2f} → {word['end_ms'] / 1000:.2f} | {word['text']}")
reviewed_project = prepared['directory']
print('Si la letra no corresponde, corrige título/artista en el paso 3 y repite la preparación.')
print('También puedes aportar un TXT corregido. Los tiempos automáticos siempre requieren revisión.')


## Generar el video
**Con voz** conserva la canción original. **Instrumental** usa el audio sin voz. **Ambos** produce dos videos.

El estilo **classic** usa un fondo sencillo y es el más rápido para empezar. **fluid** y **cosmic** añaden animación y descargan componentes del renderizador en su primera ejecución. Puedes cambiar el aspecto y repetir este paso **sin repetir la preparación**.

La revisión es importante: una transcripción automática puede equivocarse. Marca la confirmación solo después de escuchar y leer el resultado.

In [ ]:
# @title 6. Confirmar las letras y generar el video
LETRAS_REVISADAS = False # @param {type:"boolean"}
MODO = 'Con voz' # @param ['Con voz', 'Instrumental', 'Ambos']
ESTILO = 'classic' # @param ['classic', 'fluid', 'cosmic']
RESOLUCION = '720p' # @param ['720p', '1080p']
if not globals().get('prepared'):
    raise RuntimeError('Primero completa la preparación del paso 4.')
if not LETRAS_REVISADAS or globals().get('reviewed_project') != prepared['directory']:
    raise RuntimeError('Escucha y revisa las letras en el paso 5; después marca Letras revisadas.')
rendered = None
rendered = run_engine('render', {'directory': prepared['directory'], 'options': {
    'mode': {'Con voz': 'lyrics', 'Instrumental': 'karaoke', 'Ambos': 'both'}[MODO],
    'preset': ESTILO, 'resolution': RESOLUCION,
}})
print('Video terminado. Puedes verlo y descargarlo a continuación.')
for video in rendered['videos']:
    print(Path(video).name)
    display(Video(video, embed=True, width=640))


In [ ]:
# @title 7. Descargar el resultado
DESCARGAR = 'Paquete de videos (ZIP)' # @param ['Paquete de videos (ZIP)', 'Videos (MP4)', 'Proyecto editable (ZIP)']
if DESCARGAR == 'Proyecto editable (ZIP)':
    if not globals().get('prepared'):
        raise RuntimeError('Primero completa el paso 4.')
    downloads = [prepared['archive']]
else:
    if not globals().get('rendered'):
        raise RuntimeError('Primero completa el paso 6.')
    downloads = rendered['videos'] if DESCARGAR == 'Videos (MP4)' else [rendered['archive']]
for filename in downloads:
    print('Descargando:', Path(filename).name)
    files.download(filename)
print('Conserva también el proyecto editable si quieres volver a trabajar con esta canción.')


## Al terminar
Comprueba que las descargas se abren en tu equipo. Si el navegador bloquea varias descargas, elige el paquete ZIP. Después usa **Entorno de ejecución → Desconectar y eliminar entorno de ejecución** para liberar la GPU.

**Si falla un paso:** no continúes con los siguientes. Lee el último mensaje; los intentos fallidos conservan diagnósticos en la ruta indicada. Si pierdes la sesión, vuelve al paso 1. Los archivos que no descargues se pueden perder.

**Para desarrolladores:** el cuaderno utiliza `karaoke_app.notebook_runner` como adaptador del mismo motor de preparación y render que utiliza la CLI. No abre otra sesión de Colab ni modifica el Python del proveedor. Consulta `docs/notebooks.md` para distribución, arquitectura y resultados de verificación.